# BVI-Net -- Skin / Liver / Brain, one notebook

Trains the same ultra-lightweight BVI-Net (channels 4-8-8-16-16, 8 GCN nodes, real Mamba) on the
three organs from the base paper: ISIC2018 (skin), LiTS17 (liver), BraTS19 (brain).

**Run each organ as its own section -- ideally its own Kaggle session.** A session is capped at
~12 h, the liver run is the long one, and `/kaggle/working` is only saved when the session ends
normally. Run only the section(s) whose dataset is attached.

Settings: **GPU T4 x2 / P100, Internet ON** (pip + git need it).

## Datasets to attach via "+ Add Input"
1. **Skin**: `tschandl/isic2018-challenge-task1-data-segmentation`
2. **Liver**: `andrewmvd/liver-tumor-segmentation` + `andrewmvd/liver-tumor-segmentation-part-2` (attach both; the
   `javariatahir123/lits17-liver-tumor-segmentation` mirror often fails to mount)
3. **Brain** (BraTS 2019, public Kaggle mirrors -- attach ONE, check it has `HGG/` + `LGG/` case folders
   containing `*_t1/_t1ce/_t2/_flair/_seg.nii(.gz)`): `aryashah2k/brain-tumor-segmentation-brats-2019`
   (or `walidrehamnia/brain-tumor-segmentation-brats-2019-train-valid`)

The prep scripts search the mounted folders **recursively**, so the exact mount name does not matter.

## What to expect (rough estimates -- read the per-epoch time printed by train.py and adjust)
| Organ | input | slices after stride | epoch time (est.) | full run |
|---|---|---|---|---|
| Skin | 256x256 RGB | 2335 | ~5 min | ~4 h |
| Liver | 448x448 gray CT | ~1,100 train | ~8 min | ~4 h (30 ep.) |
| Brain | 240x240 4-ch MRI | ~1,900 train | ~4 min | ~2 h (30 ep.) |

If a section is too slow, raise `--slice_stride` in its prep cell or lower `--epochs`.
If liver hits `CUDA out of memory`, restart the kernel and use `--batch_size 2`.

## 0. Setup (run once per session)

In [ ]:
!pip install causal-conv1d mamba-ssm --no-build-isolation

In [ ]:
import os
if os.path.basename(os.getcwd()) != "BVI-Net-ISIC":
    !git clone https://github.com/Ghanasree-S/BVI-Net-ISIC.git
    %cd BVI-Net-ISIC
else:
    !git pull origin main
!pip install -q -r requirements.txt

In [ ]:
import os, torch
from models.fa_vssm import HAS_MAMBA
os.makedirs("checkpoints", exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- enable a GPU accelerator!")
print("Real Mamba backend:", HAS_MAMBA, "(False = slower fallback SSM; reinstall mamba-ssm above)")

# Skin has a fixed mount; liver/brain point at /kaggle/input and the prep scripts search it
# recursively (volume-N.nii / segmentation-N.nii, resp. <case>_t1/_t1ce/_t2/_flair/_seg.nii*).
SKIN_RAW_DIR  = "/kaggle/input"
LIVER_RAW_DIR = "/kaggle/input"
BRATS_RAW_DIR = "/kaggle/input"

# Which organ section(s) to run in this session. "Save & Run All" skips the others,
# so a run never crashes on a dataset that isn't attached. One organ per session is safest.
RUN = {"skin": False, "liver": True, "brain": False}
print("Sections enabled:", [k for k, v in RUN.items() if v])


In [ ]:
# Sanity-check what is attached and that each prep script will find its files.
!ls /kaggle/input
!echo "--- LiTS volumes:";  find /kaggle/input -name "volume-*.nii*" | head -3
!echo "--- BraTS LGG segs:"; find /kaggle/input -name "*_seg.nii*" -path "*LGG*" | head -3
!echo "--- ISIC training dir:"; find /kaggle/input -maxdepth 4 -iname "*Training_Input*" | head -2

## 1. Skin -- ISIC2018

Reproduces the reported result (27,312 params, Dice ~0.863): official 100-image validation set +
2335 train / 259 self-carved test.

In [ ]:
if RUN["skin"]:
    !bash data/download_isic_official_val.sh

In [ ]:
if RUN["skin"]:
    !python data/prepare_isic_a.py \
      --raw_dir {SKIN_RAW_DIR} \
      --val_raw_dir data/isic2018_val_raw \
      --out_dir data/isic2018a \
      --fraction 1.0

In [ ]:
if RUN["skin"]:
    !python train.py --dataset isic --data_dir data/isic2018a \
      --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
      --channels 4 8 8 16 16 --gcn_nodes 8 --out_dir checkpoints/skin

In [ ]:
if RUN["skin"]:
    !python evaluate.py --dataset isic --checkpoint checkpoints/skin/best.pt --data_dir data/isic2018a \
      --visualize --out_dir outputs/skin --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
if RUN["skin"]:
    import shutil
    if os.path.exists("checkpoints/skin/best.pt"):  # missing if training failed -- don't abort the other organ
        shutil.copy("checkpoints/skin/best.pt", "checkpoints/bvi_net_skin_best.pt")
        print("Saved checkpoints/bvi_net_skin_best.pt")
    else:
        print("WARNING: no skin checkpoint -- see training output above")
    if os.path.exists("outputs/skin/metrics.json"):  # test-set scores shown by the dashboard
        shutil.copy("outputs/skin/metrics.json", "checkpoints/bvi_net_skin_metrics.json")
        print("Saved checkpoints/bvi_net_skin_metrics.json")

## 2. Liver -- LiTS17

2 output channels (Liver, Tumor), 1-channel grayscale CT, 448x448. Volume-level 7:1:2 split, labeled slices only,
every 10th slice kept (neighbouring CT slices are near-duplicates; the full ~19K slices would not fit a session).
Batch size 4 because Mamba's scan memory grows with image area (448x448 = 3x the pixels of skin).

In [ ]:
if RUN["liver"]:
    !python data/prepare_lits.py \
      --raw_dir {LIVER_RAW_DIR} \
      --out_dir data/lits17a \
      --slice_stride 5

In [ ]:
if RUN["liver"]:
    !python train.py --dataset lits --data_dir data/lits17a \
      --epochs 50 --batch_size 4 --lr 0.001 --patience 15 \
      --channels 4 8 8 16 16 --gcn_nodes 8 --out_dir checkpoints/liver

In [ ]:
if RUN["liver"]:
    !python evaluate.py --dataset lits --checkpoint checkpoints/liver/best.pt --data_dir data/lits17a \
      --visualize --out_dir outputs/liver --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
if RUN["liver"]:
    import shutil
    if os.path.exists("checkpoints/liver/best.pt"):  # missing if training failed -- don't abort the other organ
        shutil.copy("checkpoints/liver/best.pt", "checkpoints/bvi_net_liver_best.pt")
        print("Saved checkpoints/bvi_net_liver_best.pt")
    else:
        print("WARNING: no liver checkpoint -- see training output above")
    if os.path.exists("outputs/liver/metrics.json"):  # test-set scores shown by the dashboard
        shutil.copy("outputs/liver/metrics.json", "checkpoints/bvi_net_liver_metrics.json")
        print("Saved checkpoints/bvi_net_liver_metrics.json")

## 3. Brain -- BraTS19 (LGG only)

3 nested output channels (WT, TC, ET), 4-channel MRI input (T1, T1ce, T2, FLAIR), 240x240. Case-level 7:1:2 split
over the 76 LGG patients, tumor-bearing slices only, every 2nd slice kept.

In [ ]:
if RUN["brain"]:
    !python data/prepare_brats.py \
      --raw_dir {BRATS_RAW_DIR} \
      --out_dir data/brats19a \
      --slice_stride 2

In [ ]:
if RUN["brain"]:
    !python train.py --dataset brats --data_dir data/brats19a \
      --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
      --channels 4 8 8 16 16 --gcn_nodes 8 --out_dir checkpoints/brain

In [ ]:
if RUN["brain"]:
    !python evaluate.py --dataset brats --checkpoint checkpoints/brain/best.pt --data_dir data/brats19a \
      --visualize --out_dir outputs/brain --channels 4 8 8 16 16 --gcn_nodes 8

In [ ]:
if RUN["brain"]:
    import shutil
    if os.path.exists("checkpoints/brain/best.pt"):  # missing if training failed -- don't abort the other organ
        shutil.copy("checkpoints/brain/best.pt", "checkpoints/bvi_net_brain_best.pt")
        print("Saved checkpoints/bvi_net_brain_best.pt")
    else:
        print("WARNING: no brain checkpoint -- see training output above")
    if os.path.exists("outputs/brain/metrics.json"):  # test-set scores shown by the dashboard
        shutil.copy("outputs/brain/metrics.json", "checkpoints/bvi_net_brain_metrics.json")
        print("Saved checkpoints/bvi_net_brain_metrics.json")

## 4. Package checkpoints + qualitative outputs for download

Run after whichever sections you completed. Download the zip from the Output panel, then drop the
`bvi_net_<organ>_best.pt` files into your local `checkpoints/` folder so `backend/app.py` can serve
real inference for each organ.

In [ ]:
import zipfile
from pathlib import Path

ckpt_dir = Path("checkpoints")
saved = sorted(p.name for p in ckpt_dir.glob("bvi_net_*_best.pt")) + sorted(p.name for p in ckpt_dir.glob("bvi_net_*_metrics.json"))
print("Checkpoints ready to download:", saved or "(none yet -- run at least one organ section above)")

if saved:
    with zipfile.ZipFile("/kaggle/working/bvi_net_checkpoints.zip", "w") as zf:
        for name in saved:
            zf.write(ckpt_dir / name, arcname=name)
        for png in sorted(Path("outputs").glob("*/sample_*.png")):   # qualitative triptychs for the report/PPT
            zf.write(png, arcname=f"outputs/{png.parent.name}/{png.name}")
    print("Wrote /kaggle/working/bvi_net_checkpoints.zip")